--- 

## **Criação de Pipeline de Extração, Limpeza, Transformação e Enriquecimento de Dados**

**Projeto**

Transações globais em várias categorias de produtos

---

## 1. Importação das bibliotecas

In [39]:
# ==============================
# 📦 IMPORTS
# ==============================
import pandas as pd

## 2. Extração dos dados

In [40]:
# ==============================
# 🔹 EXTRACT
# ==============================
def extract(caminho_arquivo):
    df = pd.read_csv(caminho_arquivo)
    return df

## 3. Transformação dos dados

In [41]:
# ==============================
# 🔹 TRANSFORM
# ==============================
def transform(df):

    # Renomear colunas
    df = df.rename(columns={
        "Transaction ID": "id_pedido",
        "Date": "data",
        "Product Category": "categoria",
        "Product Name": "produto",
        "Units Sold": "quantidade",
        "Unit Price": "preco_unitario",
        "Total Revenue": "preco_total",
        "Region": "regiao",
        "Payment Method": "forma_pagamento"
    })

    # Converter data
    df['data'] = pd.to_datetime(df['data'])

    # Criar coluna período
    df['periodo'] = df['data'].dt.to_period('M').astype(str)

    # ==============================
    # 📊 MODELAGEM SIMPLES
    # ==============================

    # Dimensões
    dim_produto = df[['produto', 'categoria']].drop_duplicates().reset_index(drop=True)
    dim_regiao = df[['regiao']].drop_duplicates().reset_index(drop=True)
    dim_tempo = df[['periodo']].drop_duplicates().reset_index(drop=True)

    # Fato
    fato_vendas = df[['quantidade', 'preco_total', 'produto', 'regiao', 'periodo']]

    return df, fato_vendas, dim_produto, dim_regiao, dim_tempo

## 4. Análise dos dados

In [42]:
# ==============================
# 🔹 ANALYTICS
# ==============================
def create_analytics(df):

    # Vendas por período
    vendas_periodo = df.groupby('periodo')['quantidade'].sum().reset_index()

    # Faturamento por período
    faturamento_periodo = df.groupby('periodo')['preco_total'].sum().reset_index()

    # Vendas por categoria
    vendas_categoria = df.groupby('categoria')['quantidade'].sum().reset_index()

    # Produtos mais vendidos
    top_produtos = df.groupby('produto')['quantidade'].sum().sort_values(ascending=False).reset_index()

    return vendas_periodo, faturamento_periodo, vendas_categoria, top_produtos

## 5. Exportação dos dados para o Excel

In [43]:
# ==============================
# 🔹 LOAD (EXPORTAÇÃO)
# ==============================
def load(df_base, fato, dim_produto, dim_regiao, dim_tempo, analytics):

    vendas_periodo, faturamento_periodo, vendas_categoria, top_produtos = analytics

    with pd.ExcelWriter('projeto_bi_vendas.xlsx') as writer:

        # Base tratada
        df_base.to_excel(writer, sheet_name='Base Limpa', index=False)

        # Modelagem
        fato.to_excel(writer, sheet_name='Fato Vendas', index=False)
        dim_produto.to_excel(writer, sheet_name='Dim Produto', index=False)
        dim_regiao.to_excel(writer, sheet_name='Dim Regiao', index=False)
        dim_tempo.to_excel(writer, sheet_name='Dim Tempo', index=False)

        # Análises
        vendas_periodo.to_excel(writer, sheet_name='Vendas por Periodo', index=False)
        faturamento_periodo.to_excel(writer, sheet_name='Faturamento', index=False)
        vendas_categoria.to_excel(writer, sheet_name='Vendas por Categoria', index=False)
        top_produtos.to_excel(writer, sheet_name='Top Produtos', index=False)

## 6. Função para executar o pipeline

In [44]:
# ==============================
# 🚀 EXECUÇÃO DO PIPELINE
# ==============================
def run_pipeline():

    caminho = "Online Sales Data.csv"

    # ETL
    df_raw = extract(caminho)
    df_base, fato, dim_produto, dim_regiao, dim_tempo = transform(df_raw)

    # Analytics
    analytics = create_analytics(df_base)

    # Load
    load(df_base, fato, dim_produto, dim_regiao, dim_tempo, analytics)

    print("✅ Pipeline executado com sucesso. Arquivo Excel gerado!")


## 7. Execução do Pipeline

In [45]:
# ==============================
# ▶ EXECUTAR
# ==============================
if __name__ == "__main__":
    run_pipeline()

✅ Pipeline executado com sucesso. Arquivo Excel gerado!


In [46]:
%reload_ext watermark
%watermark -a "Robson Silva"

Author: Robson Silva



In [47]:
%watermark --iversions

pandas: 2.3.3

